In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 23
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## IRENA Pipeline

**Source:** IRENA Renewables Capacity Statistics via Our World in Data
**Access:** Automated OWID CSV — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — IRENA_CAPACITY section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Share of electricity from renewables (%) | Environmental/climate governance | Primary tier 1 |

In [2]:
import requests
import io
import pandas as pd
from datetime import datetime

# IRENA renewable energy share via Our World in Data
# Share of electricity from renewables — country-level, 1985-present
OWID_IRENA_URL = "https://ourworldindata.org/grapher/share-electricity-renewables.csv?v=1&csvType=full&useColumnShortNames=false"

print("Downloading IRENA renewable share data from Our World in Data...")
response = requests.get(OWID_IRENA_URL, timeout=30)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024:.1f}KB")

irena_raw = pd.read_csv(io.StringIO(response.text))
print(f"\nShape: {irena_raw.shape}")
print(f"Columns: {list(irena_raw.columns)}")
print(f"Years: {irena_raw['Year'].min()} — {irena_raw['Year'].max()}")
print(f"Entities: {irena_raw['Entity'].nunique()}")
print(irena_raw.head(3))

Status: 200, Size: 216.0KB

Shape: (7872, 4)
Columns: ['Entity', 'Code', 'Year', 'Renewables']
Years: 1985 — 2025
Entities: 251
          Entity Code  Year  Renewables
0  ASEAN (Ember)  NaN  2000   19.334143
1  ASEAN (Ember)  NaN  2001   19.055025
2  ASEAN (Ember)  NaN  2002   17.666613


In [3]:
# Filter and rename columns
irena = irena_raw.copy()
irena = irena.rename(columns={
    'Entity':      'country_name',
    'Code':        'country_code',
    'Year':        'year',
    'Renewables':  'irena_renewables_share_pct',
})

# Drop rows with no country code — regional aggregates
irena = irena[irena['country_code'].notna() & (irena['country_code'] != '')].copy()

# Filter to framework start year
irena = irena[irena['year'] >= FRAMEWORK_START_YEAR].copy()
irena = irena.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {irena.shape}")
print(f"Years: {irena['year'].min()} — {irena['year'].max()}")
print(f"Countries: {irena['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (irena.isnull().sum() / len(irena) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(irena.head(3))

Shape: (6600, 4)
Years: 1990 — 2025
Countries: 226

Missing values (%):
Series([], dtype: float64)
  country_name country_code  year  irena_renewables_share_pct
0  Afghanistan          AFG  2000                   64.583336
1  Afghanistan          AFG  2001                   72.463770
2  Afghanistan          AFG  2002                   78.873245


In [4]:
# Derive metadata from data — no hardcoding
latest_year = str(int(irena['year'].max()))
data_as_of_date = latest_year

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "irena_clean.csv")
irena.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {irena.shape}")

# Update download log
update_entry(
    "IRENA_CAPACITY",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="irena_clean.csv",
    latest_available_version=latest_year,
    notes="Share of electricity from renewables (%). Downloaded via OWID. Original source: IRENA/Ember. Coverage: 1990-2025, 226 countries."
)

print_entry("IRENA_CAPACITY")

Written: /Users/boulanger/Documents/governance-framework/data/processed/irena_clean.csv
Shape: (6600, 4)
[download_log] Updated entry for IRENA_CAPACITY
  source_id: IRENA_CAPACITY
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2025
  local_filename: irena_clean.csv
  latest_available_version: 2025
  no_update_reason: nan
  notes: Share of electricity from renewables (%). Downloaded via OWID. Original source: IRENA/Ember. Coverage: 1990-2025, 226 countries.
